# SigFlow v4 research runner

This is the Run-All entry point for the canonical `sigflow_v4_research.py` implementation. The default is the proper real-data development run. Change only the controls in the next cell, restart the kernel, and choose **Run All**.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
EXPECTED_ENV = (PROJECT_ROOT / ".venv").resolve()
if Path(sys.prefix).resolve() != EXPECTED_ENV:
    raise RuntimeError(
        f"Wrong notebook kernel: {sys.executable}. Select the .venv kernel at "
        f"{PROJECT_ROOT / '.venv/bin/python'} and restart the notebook."
    )

MPL_CACHE = PROJECT_ROOT / ".matplotlib-cache"
MPL_CACHE.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE))

import torch
import sigflow_v4_research as sf

# pipeline_test | development | rolling_research
MODE = "development"
RUN_ABLATIONS = False
RUN_TICKER_MODEL_COMPARISON = False
SHOW_PLOTS = False
DISPLAY_TABLES = False
SAVE_ARTIFACTS = True

if MODE not in {"pipeline_test", "development", "rolling_research"}:
    raise ValueError(f"Unknown MODE={MODE!r}")
if RUN_ABLATIONS and RUN_TICKER_MODEL_COMPARISON:
    raise ValueError("Choose ablations or ticker comparison, not both.")
if MODE != "development" and (RUN_ABLATIONS or RUN_TICKER_MODEL_COMPARISON):
    raise ValueError("Ablations and ticker comparison require MODE='development'.")

PROFILE = "smoke" if MODE == "pipeline_test" else "gpu_long"
cfg = sf.config_for_profile(PROFILE)
print("Python:", sys.version.split()[0], "| executable:", sys.executable)
print("Mode:", MODE, "| profile:", PROFILE)
print("CUDA available:", torch.cuda.is_available())
if MODE != "pipeline_test" and not torch.cuda.is_available():
    print("WARNING: the full five-seed research run will use CPU and take a long time.")
cfg


In [ ]:
import time

RUN_STARTED = time.perf_counter()
if MODE == "rolling_research":
    RESULTS = sf.run_multi_horizon_rolling_research(
        cfg, show_plots=SHOW_PLOTS, save_artifacts=SAVE_ARTIFACTS
    )
else:
    DATASET_STARTED = time.perf_counter()
    DATASET = sf.build_market_dataset(cfg)
    print(f"Dataset construction completed in {(time.perf_counter() - DATASET_STARTED) / 60:.2f} minutes.")

    if RUN_TICKER_MODEL_COMPARISON:
        RESULTS = sf.run_separate_ticker_validation_comparison(
            cfg, DATASET, save_artifacts=SAVE_ARTIFACTS
        )
    else:
        sf.RUN_GROUP_ABLATIONS = bool(RUN_ABLATIONS)
        sf.RUN_INDIVIDUAL_FEATURE_ABLATIONS = False
        if RUN_ABLATIONS:
            ABLATION_SUMMARY = sf.run_configured_validation_ablations(cfg, DATASET)
            cfg = sf.apply_validation_selected_gate(cfg, ABLATION_SUMMARY)
        RESULTS = sf.main(
            cfg,
            dataset=DATASET,
            evaluation_split="validation",
            show_plots=SHOW_PLOTS,
            display_tables=DISPLAY_TABLES,
            save_artifacts=SAVE_ARTIFACTS,
        )

print(f"Total Run-All time: {(time.perf_counter() - RUN_STARTED) / 60:.2f} minutes.")


## One-time prospective test

Do not run the final test from this notebook. It is deliberately locked until data through 2028-09-15 exist. Use this notebook only for pipeline testing, development validation, ablations, ticker comparisons, and rolling-origin research.